For demonstration, we'll use a dataset of events that occured in American Football games.

The first thing to do when you get a new dataset is take a look at some of it. This lets you see that it all read in correctly and gives an idea of what's going on with the data.

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np

nfl_data = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "maxhorowitz/nflplaybyplay2009to2016",
    "NFL Play by Play 2009-2017 (v4).csv",
    pandas_kwargs={
          "compression": "zip",
          "encoding": "utf-8-sig",
          "low_memory": False,
      },
  )

np.random.seed(0)
nfl_data.head()

/home/kiwi/projects/self-learn-codes/student_versions/kaggle_tutorials/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,Date,GameID,Drive,qtr,down,time,TimeUnder,TimeSecs,PlayTimeDiff,SideofField,...,yacEPA,Home_WP_pre,Away_WP_pre,Home_WP_post,Away_WP_post,Win_Prob,WPA,airWPA,yacWPA,Season
0,2009-09-10,2009091000,1,1,NaN,15:00,15,3600.0,0.0,TEN,...,NaN,0.485675,0.514325,0.546433,0.453567,0.485675,0.060758,NaN,NaN,2009
1,2009-09-10,2009091000,1,1,1.0,14:53,15,3593.0,7.0,PIT,...,1.146076,0.546433,0.453567,0.551088,0.448912,0.546433,0.004655,-0.032244,0.036899,2009
2,2009-09-10,2009091000,1,1,2.0,14:16,15,3556.0,37.0,PIT,...,NaN,0.551088,0.448912,0.510793,0.489207,0.551088,-0.040295,NaN,NaN,2009
3,2009-09-10,2009091000,1,1,3.0,13:35,14,3515.0,41.0,PIT,...,-5.031425,0.510793,0.489207,0.461217,0.538783,0.510793,-0.049576,0.106663,-0.156239,2009
4,2009-09-10,2009091000,1,1,4.0,13:27,14,3507.0,8.0,PIT,...,NaN,0.461217,0.538783,0.558929,0.441071,0.461217,0.097712,NaN,NaN,2009


# How many missing data points do we have?

In [2]:
nfl_data.isnull().sum().sort_values(ascending=False)

DefTwoPoint          407664
BlockingPlayer       407571
TwoPointConv         407083
ChalReplayResult     404286
RecFumbPlayer        403315
                      ...  
HomeTeam                  0
Timeout_Indicator         0
TwoPoint_Prob             0
ExPoint_Prob              0
Season                    0
Length: 102, dtype: int64

Now that we see there are lots of missing values, we can calculate the extent of missing values relative to all values in the given columns.

In [3]:
nfl_data.isnull().sum().sort_values(ascending=False) / nfl_data.shape[0]

DefTwoPoint          0.999941
BlockingPlayer       0.999713
TwoPointConv         0.998516
ChalReplayResult     0.991655
RecFumbPlayer        0.989274
                       ...   
HomeTeam             0.000000
Timeout_Indicator    0.000000
TwoPoint_Prob        0.000000
ExPoint_Prob         0.000000
Season               0.000000
Length: 102, dtype: float64

We can also see how many values are missing in the entirety of this DataFrame, since some columns are rather complete as well.

In [4]:
nfl_data.isnull().sum().sum() / np.prod(nfl_data.shape)

# The first .sum() operates down each column, and the second .sum() adds those column counts
# For total number of elements, we simply multiply by the shape of nfl_data, which is a tuple of (rows, columns). We can use np.prod to multiply the two values together.

np.float64(0.2766722370547874)

# Figure out why the data is missing

One of the most important questions you can ask yourself to help figure this out is this:

```text
Is this value missing because it wasn't recorded or because it doesn't exist?
```

- If a value is missing becuase **it doesn't exist** (like the height of the oldest child of someone who doesn't have any children) then it doesn't make sense to try and guess what it might be. These values you probably do want to keep as NaN.
- On the other hand, if a value is missing because it **wasn't recorded**, then you can try to guess what it might have been based on the other values in that column and row. This is called imputation.

Let's now see what's missing by just looking at the first 10 columns.

In [5]:
nfl_data.isnull().sum()[0:10]

Date                0
GameID              0
Drive               0
qtr                 0
down            61154
time              224
TimeUnder           0
TimeSecs          224
PlayTimeDiff      444
SideofField       528
dtype: int64

Some columns, such as the number of seconds left in the game when the play was made, were probably not recorded and it would make sense to try & guess them.

Some other columns, like `PenalizedTeam`, probably don't exist simply because there was no penalty, so it would make sense to leave it empty or replace `NaN` with neither.

# Drop missing values

If you're in a hurry or don't have a reason to figure out why your values are missing, one option you have is to just remove any rows or columns that contain missing values.

(Note: I don't generally recommend this approch for important projects! It's usually worth it to take the time to go through your data and really look at all the columns with missing values one-by-one to really get to know your dataset.)

For this exercise, let's go ahead and drop ALL of the the missing values (`NaN`) via `dropna()`.

In [6]:
nfl_data.dropna()

# Defaults to axis=0
# Removes rows containing at least one missing value (NaN)
# Returns a new DataFrame; it does not modify nfl_data unless assigned back or used with inplace=Tru

,Date,GameID,Drive,qtr,down,time,TimeUnder,TimeSecs,PlayTimeDiff,SideofField,...,yacEPA,Home_WP_pre,Away_WP_pre,Home_WP_post,Away_WP_post,Win_Prob,WPA,airWPA,yacWPA,Season


However, this actually removed all our data. This is because every row in our dataset had at least one missing value.

We might have better luck removing all the columns that have at least one missing value instead.

In [ ]:
columns_with_na_dropped = nfl_data.dropna(axis=1)
columns_with_na_dropped.head()

# axis=1 means operate on columns
# Removes any column that contains at least one missing value
# Saves the result as columns_with_na_dropped
# .head() displays its first five rows

,Date,GameID,Drive,qtr,TimeUnder,ydstogo,ydsnet,PlayAttempted,Yards.Gained,sp,...,AwayTeam,Timeout_Indicator,posteam_timeouts_pre,HomeTimeouts_Remaining_Pre,AwayTimeouts_Remaining_Pre,HomeTimeouts_Remaining_Post,AwayTimeouts_Remaining_Post,ExPoint_Prob,TwoPoint_Prob,Season
0,2009-09-10,2009091000,1,1,15,0,0,1,39,0,...,TEN,0,3,3,3,3,3,0.0,0.0,2009
1,2009-09-10,2009091000,1,1,15,10,5,1,5,0,...,TEN,0,3,3,3,3,3,0.0,0.0,2009
2,2009-09-10,2009091000,1,1,15,5,2,1,-3,0,...,TEN,0,3,3,3,3,3,0.0,0.0,2009
3,2009-09-10,2009091000,1,1,14,8,2,1,0,0,...,TEN,0,3,3,3,3,3,0.0,0.0,2009
4,2009-09-10,2009091000,1,1,14,8,2,1,0,0,...,TEN,0,3,3,3,3,3,0.0,0.0,2009


In [8]:
# Just how much data did we lose?

print("Columns in original dataset: %d" % nfl_data.shape[1])
print("Columns with na's dropped: %d" % columns_with_na_dropped.shape[1])

Columns in original dataset: 102
Columns with na's dropped: 37


# Filling in missing values automatically

For this next bit, I'm getting a small sub-section of the NFL data so that it will print well.

In [ ]:
# Get a small subset of the NFL dataset

subset_nfl_data = nfl_data.loc[:, 'EPA': 'Season'].head() # For all rows, select columns from 'EPA' to 'Season', and then take the first five rows
subset_nfl_data

,EPA,airEPA,yacEPA,Home_WP_pre,Away_WP_pre,Home_WP_post,Away_WP_post,Win_Prob,WPA,airWPA,yacWPA,Season
0,2.014474,NaN,NaN,0.485675,0.514325,0.546433,0.453567,0.485675,0.060758,NaN,NaN,2009
1,0.077907,-1.068169,1.146076,0.546433,0.453567,0.551088,0.448912,0.546433,0.004655,-0.032244,0.036899,2009
2,-1.402760,NaN,NaN,0.551088,0.448912,0.510793,0.489207,0.551088,-0.040295,NaN,NaN,2009
3,-1.712583,3.318841,-5.031425,0.510793,0.489207,0.461217,0.538783,0.510793,-0.049576,0.106663,-0.156239,2009
4,2.097796,NaN,NaN,0.461217,0.538783,0.558929,0.441071,0.461217,0.097712,NaN,NaN,2009


We can use the Panda's `fillna()` function to fill in missing values in a dataframe for us. Here, I'm saying that I would like to replace all the `NaN` values with 0.

In [10]:
subset_nfl_data.fillna(0)

,EPA,airEPA,yacEPA,Home_WP_pre,Away_WP_pre,Home_WP_post,Away_WP_post,Win_Prob,WPA,airWPA,yacWPA,Season
0,2.014474,0.000000,0.000000,0.485675,0.514325,0.546433,0.453567,0.485675,0.060758,0.000000,0.000000,2009
1,0.077907,-1.068169,1.146076,0.546433,0.453567,0.551088,0.448912,0.546433,0.004655,-0.032244,0.036899,2009
2,-1.402760,0.000000,0.000000,0.551088,0.448912,0.510793,0.489207,0.551088,-0.040295,0.000000,0.000000,2009
3,-1.712583,3.318841,-5.031425,0.510793,0.489207,0.461217,0.538783,0.510793,-0.049576,0.106663,-0.156239,2009
4,2.097796,0.000000,0.000000,0.461217,0.538783,0.558929,0.441071,0.461217,0.097712,0.000000,0.000000,2009


I could also be a bit more savvy and replace missing values with whatever value comes directly after it in the same column. 

This makes a lot of sense for datasets where the observations have some sort of logical order to them.

In [ ]:
subset_nfl_data.fillna(method='bfill', axis=0).fillna(0) # Forward fill then backward fill

# fillna(method='bfill', axis=0) fills each missing value with the next non-missing value below it in the same column (“backward fill”)
# .fillna(0) replaces any remaining missing values—typically those at the bottom of a column—with 0

# Example:

# Before:        After:
# 5              5
# NaN     →      8
# 8              8
# NaN            0